<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 1 — *Hello RL World*: Procesos de Decisión de Markov en R

### Versión adaptada a Google Colab

**Curso:** Aprendizaje por Refuerzo: Fundamentos y Aplicaciones
**Institución:** Universidad Austral — Facultad de Ingeniería, Posgrados
**Docente:** Dr. Darío Ezequiel Díaz
**Fecha:** 5 de mayo de 2026

---

> ⚠️ **Antes de comenzar.** Configure el runtime en R: **Entorno de ejecución → Cambiar tipo de entorno → R → Guardar**. Una vez confirmado el runtime, ejecute las celdas en orden.

### Propósito del cuaderno

Este cuaderno cumple dos funciones complementarias. Por un lado, **materializa en código R** la formalización matemática del Proceso de Decisión de Markov $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ presentada en la sesión sincrónica. Por el otro, **introduce el patrón de interoperación R–Python** mediante `reticulate`, recurso que utilizaremos a lo largo del curso para acceder al ecosistema Gymnasium sin renunciar al rico instrumental estadístico nativo de R.

El recorrido procede del particular al general: comenzaremos implementando un GridWorld desde cero —donde cada componente del MDP queda explícitamente expuesto mediante una clase `R6`— para luego acceder a un entorno canónico de Gymnasium (CartPole-v1) y observar cómo la abstracción algorítmica oculta esos mismos componentes detrás de una API uniforme. Finalmente, un análisis estadístico Monte Carlo de retornos bajo política aleatoria nos permitirá fijar las herramientas inferenciales que emplearemos sistemáticamente en clases ulteriores.

> **Diferencia respecto del cuaderno local.** Esta versión Colab no utiliza el entorno conda `rl-docencia` —que sólo existe en su máquina—. En su lugar, instala `gymnasium` directamente en la Python que provee el runtime de Colab. El resto del contenido es idéntico al del cuaderno local.

## 0. Configuración del entorno computacional

La configuración para Colab procede en tres pasos: cargar los paquetes R necesarios (idealmente desde el script de setup personal en Drive), instalar `gymnasium` en la Python del runtime, y verificar que todo funcione.

In [ ]:
# --- Paso 1: paquetes R ---
# Si dispone de su script de setup personal en Drive, sourcearlo.
# En caso contrario, instalar los paquetes mínimos necesarios para esta clase.

ruta_setup_personal <- "/content/drive/MyDrive/R_Colab/setup_R_colab.R"

if (file.exists(ruta_setup_personal)) {
  cat("Cargando setup personal desde Drive...\n")
  source(ruta_setup_personal)
} else {
  cat("Setup personal no encontrado. Instalando paquetes mínimos para Clase 1...\n")
  paquetes_clase1 <- c("R6", "ggplot2", "patchwork", "reticulate")
  faltantes <- paquetes_clase1[!sapply(paquetes_clase1, function(p) requireNamespace(p, quietly = TRUE))]
  if (length(faltantes) > 0) {
    install.packages(faltantes, quiet = TRUE)
  }
  cat("Paquetes verificados.\n")
}

library(reticulate)
library(R6)
library(ggplot2)

In [ ]:
# --- Paso 2: instalar gymnasium en la Python de Colab ---
# A diferencia del entorno local con conda, Colab usa una Python única.
# Instalamos gymnasium una sola vez por sesión.

if (!py_module_available("gymnasium")) {
  cat("Instalando gymnasium en el runtime de Colab (~30 segundos)...\n")
  py_install("gymnasium", pip = TRUE)
  cat("Listo.\n")
} else {
  cat("gymnasium ya disponible en el runtime.\n")
}

In [ ]:
# --- Paso 3: importación y verificación de versiones ---
gym <- import("gymnasium")
np  <- import("numpy")

# Reproducibilidad: semilla global compartida entre R y Python
SEMILLA <- 42L
set.seed(SEMILLA)

cat("─── Entorno verificado ───\n")
cat(sprintf("R version : %s\n",   R.version.string))
cat(sprintf("Python    : %s\n",   as.character(py_version())))
cat(sprintf("Gymnasium : %s\n",   gym$`__version__`))
cat(sprintf("NumPy     : %s\n",   np$`__version__`))
cat(sprintf("Semilla   : %d\n",   SEMILLA))

> **Sobre los tipos enteros en `reticulate`.** Python distingue rigurosamente entre enteros y números de coma flotante; muchas funciones de Gymnasium exigen enteros estrictos. En R, debemos forzar la conversión usando el sufijo `L` (por ejemplo, `42L`) o la función `as.integer()`. Esta sutileza, aparentemente menor, evita errores opacos cuando se invoca código Python desde R.

## 1. El MDP en código: GridWorld desde cero

### 1.1 Recordatorio formal

Recuérdese que un Proceso de Decisión de Markov es una quíntupla
$$
\mathcal{M} = (\mathcal{S},\, \mathcal{A},\, P,\, R,\, \gamma)
$$
donde $\mathcal{S}$ es el espacio de estados, $\mathcal{A}$ el espacio de acciones, $P(s' \mid s, a)$ la dinámica de transición, $R(s, a)$ la función de recompensa y $\gamma \in [0,1]$ el factor de descuento.

Implementaremos un **GridWorld** $4 \times 4$ con la siguiente especificación, idéntica a la presentada en la sesión sincrónica:

- $\mathcal{S} = \{(i, j) : i, j \in \{0, 1, 2, 3\}\} \setminus \text{muros}$, con muros en $(1, 1)$ y $(2, 1)$.
- Estado inicial: $s_0 = (0, 0)$. Estado terminal (meta): $s_g = (3, 3)$.
- $\mathcal{A} = \{\uparrow, \downarrow, \leftarrow, \rightarrow\}$ codificadas como $\{0, 1, 2, 3\}$.
- $P$ es **determinista** en esta versión inicial.
- $R(s, a) = -1$ por cada paso, $+10$ al alcanzar la meta.
- $\gamma = 0.95$.

### 1.2 Implementación mediante `R6`

Optamos por la clase de objetos `R6` —semánticamente próxima a las clases de Python— para preservar el paralelismo conceptual con el cuaderno hermano. Las funciones miembro materializan, una a una, los componentes del MDP.

In [ ]:
GridWorld <- R6Class("GridWorld",

  public = list(

    # ---- Atributos ----
    dim              = NULL,
    muros            = NULL,
    inicio           = NULL,
    meta             = NULL,
    recompensa_paso  = NULL,
    recompensa_meta  = NULL,
    gamma            = NULL,
    S                = NULL,
    A                = NULL,
    s_actual         = NULL,

    # Codificación de acciones
    ARRIBA    = 0L,
    ABAJO     = 1L,
    IZQUIERDA = 2L,
    DERECHA   = 3L,
    NOMBRE_ACCION = c("0" = "↑", "1" = "↓", "2" = "←", "3" = "→"),

    # ---- Constructor ----
    initialize = function(dim = 4L,
                          muros = list(c(1, 1), c(2, 1)),
                          inicio = c(0, 0),
                          meta = c(3, 3),
                          recompensa_paso = -1,
                          recompensa_meta = 10,
                          gamma = 0.95) {

      self$dim             <- dim
      self$muros           <- muros
      self$inicio          <- inicio
      self$meta            <- meta
      self$recompensa_paso <- recompensa_paso
      self$recompensa_meta <- recompensa_meta
      self$gamma           <- gamma

      # Construcción del espacio de estados (excluye muros)
      coords <- expand.grid(i = 0:(dim - 1), j = 0:(dim - 1))
      es_muro <- apply(coords, 1, function(fila) {
        any(sapply(muros, function(m) all(m == fila)))
      })
      self$S <- as.matrix(coords[!es_muro, ])
      self$A <- c(self$ARRIBA, self$ABAJO, self$IZQUIERDA, self$DERECHA)
    },

    # ---- Componentes del MDP ----
    es_muro = function(s) {
      any(sapply(self$muros, function(m) all(m == s)))
    },

    transicion = function(s, a) {
      # Dinámica determinista P(s' | s, a)
      i <- s[1]; j <- s[2]
      candidato <- if      (a == self$ARRIBA)    c(i,     j + 1)
                   else if (a == self$ABAJO)     c(i,     j - 1)
                   else if (a == self$IZQUIERDA) c(i - 1, j)
                   else if (a == self$DERECHA)   c(i + 1, j)
                   else stop("Acción inválida: ", a)

      fuera_grilla <- candidato[1] < 0 || candidato[1] >= self$dim ||
                      candidato[2] < 0 || candidato[2] >= self$dim
      if (fuera_grilla || self$es_muro(candidato)) return(s)
      candidato
    },

    recompensa = function(s, a, s_prima) {
      if (all(s_prima == self$meta)) self$recompensa_meta
      else                            self$recompensa_paso
    },

    es_terminal = function(s) all(s == self$meta),

    # ---- API tipo Gymnasium ----
    reset = function() {
      self$s_actual <- self$inicio
      self$s_actual
    },

    step = function(a) {
      s <- self$s_actual
      s_prima <- self$transicion(s, a)
      r       <- self$recompensa(s, a, s_prima)
      self$s_actual <- s_prima
      list(s_prima = s_prima, r = r, terminado = self$es_terminal(s_prima))
    }
  )
)

# Instanciación
gw <- GridWorld$new()
cat("|S| =", nrow(gw$S), "estados (excluyendo muros)\n")
cat("|A| =", length(gw$A), "acciones\n")
cat("gamma =", gw$gamma, "\n\n")
cat("Espacio de estados:\n")
print(gw$S)

### 1.3 Visualización del entorno

Construimos un gráfico con `ggplot2` que reproduce la estética institucional empleada en la presentación.

In [ ]:
# Paleta institucional Universidad Austral
PALETA <- c(
  navy        = "#1E3A5F",
  navyLight   = "#3D5A85",
  orange      = "#D86A2C",
  orangeLight = "#F2A06B",
  teal        = "#A8DADC",
  gris        = "#4A4A4A"
)

# Construcción del data frame de celdas
celdas <- expand.grid(i = 0:3, j = 0:3)
celdas$tipo <- "vacia"
celdas$tipo[celdas$i == 1 & celdas$j == 1] <- "muro"
celdas$tipo[celdas$i == 2 & celdas$j == 1] <- "muro"
celdas$tipo[celdas$i == 0 & celdas$j == 0] <- "inicio"
celdas$tipo[celdas$i == 3 & celdas$j == 3] <- "meta"

etiquetas <- data.frame(
  i = c(0, 3),
  j = c(0, 3),
  texto = c("S", "G")
)

options(repr.plot.width = 5, repr.plot.height = 5)

ggplot(celdas, aes(x = i, y = j, fill = tipo)) +
  geom_tile(color = PALETA["navy"], linewidth = 1) +
  geom_text(data = etiquetas, aes(x = i, y = j, label = texto),
            inherit.aes = FALSE, size = 6, fontface = "bold") +
  scale_fill_manual(
    values = c(vacia  = "white",
               muro   = unname(PALETA["gris"]),
               inicio = unname(PALETA["teal"]),
               meta   = unname(PALETA["orangeLight"])),
    guide = "none"
  ) +
  coord_fixed() +
  labs(title = "GridWorld 4×4 — entorno didáctico") +
  theme_void() +
  theme(plot.title = element_text(color = PALETA["navy"], hjust = 0.5,
                                   face = "bold", size = 13))

### 1.4 Verificación de la propiedad de Markov

Recuérdese que la propiedad de Markov establece
$$
\mathbb{P}(S_{t+1} \mid S_0, A_0, \ldots, S_t, A_t) \;=\; \mathbb{P}(S_{t+1} \mid S_t, A_t).
$$

En nuestro `GridWorld`, el método `transicion(s, a)` depende **exclusivamente** del par $(s, a)$ actual; no recibe ni consulta el historial. Esto garantiza por construcción la propiedad. Verifiquémoslo numéricamente: si invocamos `transicion(s, a)` repetidas veces con el mismo argumento, debemos obtener idéntico resultado, independientemente de cuántas llamadas previas hayamos efectuado.

In [ ]:
s_actual  <- c(1, 2)
a_actual  <- gw$DERECHA

resultados <- replicate(5, gw$transicion(s_actual, a_actual), simplify = FALSE)
todos_iguales <- length(unique(resultados)) == 1L

cat(sprintf("T((%d,%d), %s) = (%d,%d)\n",
            s_actual[1], s_actual[2],
            gw$NOMBRE_ACCION[as.character(a_actual)],
            resultados[[1]][1], resultados[[1]][2]))
cat("Resultado constante en", length(resultados), "llamadas:",
    todos_iguales, "\n")
cat("Propiedad de Markov verificada por construcción.\n")

## 2. Gymnasium: el ecosistema estándar de entornos RL

### 2.1 Filosofía de la API

[Gymnasium](https://gymnasium.farama.org/) es la evolución mantenida del histórico OpenAI Gym. Provee una **interfaz uniforme** para entornos heterogéneos, desde mundos discretos como FrozenLake hasta simulaciones físicas como MuJoCo. Esta uniformidad es deliberada: permite escribir algoritmos agnósticos al entorno y, recíprocamente, comparar entornos bajo un mismo agente.

La API se reduce conceptualmente a tres llamadas, cuya signatura desde R es muy próxima —pero no idéntica— a la de Python:

| Llamada (R via `reticulate`) | Propósito |
|---|---|
| `env$reset(seed = 42L)` | Inicia un nuevo episodio; devuelve `list(obs, info)`. |
| `env$step(a)` | Aplica la acción $a$; devuelve `list(obs, r, terminated, truncated, info)`. |
| `env$close()` | Libera recursos del simulador. |

### 2.2 CartPole-v1 como MDP

Trabajaremos con [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), problema canónico de control. Físicamente, consiste en un péndulo invertido montado sobre un carro que se desplaza por un riel; el agente debe mantener el péndulo en posición vertical aplicando fuerzas laterales al carro.

Su formalización como MDP:

- $\mathcal{S} \subset \mathbb{R}^4$, donde $s = (x, \dot{x}, \theta, \dot{\theta})$ son posición y velocidad del carro, ángulo y velocidad angular del péndulo.
- $\mathcal{A} = \{0, 1\}$: empujar a la izquierda o a la derecha.
- $P$ es la dinámica determinista del sistema mecánico (con condición inicial aleatoria).
- $R(s, a) = +1$ por cada paso en que el péndulo se mantenga en pie.
- Episodio termina si $|\theta| > 12°$ o $|x| > 2.4$ (caída) o se cumplen 500 pasos (truncamiento).

In [ ]:
env <- gym$make("CartPole-v1")

# unlist convierte tuplas Python (listas R) en vectores atómicos imprimibles
forma <- unlist(env$observation_space$shape)
cota_inf <- env$observation_space$low
cota_sup <- env$observation_space$high

cat("=== Inspección del entorno CartPole-v1 ===\n\n")
cat(sprintf("Espacio de observación   : %s\n", py_str(env$observation_space)))
cat(sprintf("  Dimensiones            : %s\n", paste(forma, collapse = " × ")))
cat(sprintf("  Cota inferior          : [%s]\n",
            paste(format(cota_inf, digits = 4), collapse = ", ")))
cat(sprintf("  Cota superior          : [%s]\n",
            paste(format(cota_sup, digits = 4), collapse = ", ")))
cat(sprintf("\nEspacio de acciones      : %s\n", py_str(env$action_space)))
cat(sprintf("  Cardinalidad           : %d\n", as.integer(env$action_space$n)))
cat(sprintf("\nLímite de pasos por episodio: %d\n",
            as.integer(env$spec$max_episode_steps)))

### 2.3 Mapeo formal MDP $\leftrightarrow$ API

Conviene fijar la correspondencia entre los símbolos matemáticos y los nombres del código, pues será invocada implícitamente en todos los algoritmos del curso:

| Notación matemática | Nombre en código (R) |
|:-:|:-:|
| $s_0 \sim \mu_0$ | `res <- env$reset(); obs <- res[[1]]` |
| $s_{t+1} \sim P(\cdot \mid s_t, a_t)$ | primer elemento de `env$step(a)` |
| $r_{t+1} = R(s_t, a_t, s_{t+1})$ | segundo elemento de `env$step(a)` |
| $\mathbb{1}\{s \text{ es terminal}\}$ | `terminated` (tercer elemento) |
| $a_t \in \mathcal{A}$ | argumento entero de `env$step` |

Examinemos un único paso para fijar la sintaxis:

In [ ]:
res <- env$reset(seed = SEMILLA)
obs <- as.numeric(res[[1]])  # conversión explícita: numpy array → R numeric

cat(sprintf("Estado inicial s_0 = [%s]\n",
            paste(round(obs, 4), collapse = ", ")))
cat("  Componentes : (x, x_punto, theta, theta_punto)\n")
cat("  Significado : (posición, velocidad, ángulo, vel. angular)\n\n")

# Aplicamos una acción arbitraria: empujar a la derecha
a <- 1L
paso <- env$step(a)
obs_sig    <- as.numeric(paso[[1]])
recompensa <- as.numeric(paso[[2]])
terminated <- as.logical(paso[[3]])
truncated  <- as.logical(paso[[4]])

cat(sprintf("Acción aplicada: a = %d\n", a))
cat(sprintf("Estado s_1     : [%s]\n", paste(round(obs_sig, 4), collapse = ", ")))
cat(sprintf("Recompensa r_1 : %g\n", recompensa))
cat(sprintf("Terminado      : %s\n", terminated))
cat(sprintf("Truncado       : %s\n", truncated))

## 3. Primera política: el agente aleatorio

### 3.1 Definición formal

Una **política** $\pi$ es una distribución condicional sobre acciones dado el estado:
$$
\pi(a \mid s) \;=\; \mathbb{P}(A_t = a \mid S_t = s).
$$

La política aleatoria uniforme se define por
$$
\pi_{\text{rand}}(a \mid s) \;=\; \frac{1}{|\mathcal{A}|} \quad \forall\, s \in \mathcal{S},\; \forall\, a \in \mathcal{A}.
$$

Es la política más simple concebible: el agente ignora el estado y selecciona acciones equiprobablemente. Constituye una **línea de base** indispensable: cualquier algoritmo que diseñemos en clases sucesivas deberá superar consistentemente su rendimiento.

In [ ]:
politica_aleatoria <- function(estado, env) {
  # pi_rand: muestrea uniformemente del espacio de acciones.
  # El argumento `estado` se ignora deliberadamente.
  n_acciones <- as.integer(env$action_space$n)
  as.integer(sample.int(n_acciones, size = 1) - 1L)
}

rollout <- function(env, politica, max_pasos = 500L, semilla = NULL) {
  # Ejecuta un episodio completo bajo la política dada.
  res <- if (!is.null(semilla)) env$reset(seed = as.integer(semilla))
         else                    env$reset()
  estado <- as.numeric(res[[1]])  # conversión explícita: numpy array → R numeric

  estados     <- list(estado)
  acciones    <- integer(0)
  recompensas <- numeric(0)

  for (t in seq_len(max_pasos)) {
    accion <- politica(estado, env)
    paso <- env$step(accion)
    estado     <- as.numeric(paso[[1]])
    r          <- as.numeric(paso[[2]])
    terminated <- as.logical(paso[[3]])
    truncated  <- as.logical(paso[[4]])

    estados[[length(estados) + 1L]] <- estado
    acciones    <- c(acciones, accion)
    recompensas <- c(recompensas, r)

    if (terminated || truncated) break
  }

  list(estados               = estados,
       acciones              = acciones,
       recompensas           = recompensas,
       longitud              = length(recompensas),
       retorno_no_descontado = sum(recompensas))
}

# Ejecución de un único episodio
traj <- rollout(env, politica_aleatoria, semilla = SEMILLA)

cat("Longitud del episodio   :", traj$longitud, "pasos\n")
cat("Retorno (no descontado) : G_0 =", traj$retorno_no_descontado, "\n\n")
cat("Primeros 5 pasos de la trayectoria:\n")
for (t in seq_len(min(5L, traj$longitud))) {
  s <- round(traj$estados[[t]], 3)
  a <- traj$acciones[t]
  r <- traj$recompensas[t]
  cat(sprintf("  t=%d: s=(%s), a=%d, r=%.1f\n",
              t - 1L, paste(s, collapse = ", "), a, r))
}

### 3.2 Cálculo del retorno descontado

El **retorno descontado** desde el paso $t$ se define
$$
G_t \;=\; \sum_{k=0}^{T-t-1} \gamma^k\, R_{t+k+1}.
$$

Implementemos su cálculo y comparemos para distintos valores de $\gamma$, ilustrando el papel del factor de descuento como ponderador temporal.

In [ ]:
retorno_descontado <- function(recompensas, gamma) {
  k <- seq_along(recompensas) - 1L
  sum(gamma^k * recompensas)
}

cat("Retorno descontado de la trayectoria anterior, para distintos gamma:\n\n")
for (g in c(0.0, 0.5, 0.9, 0.99, 1.0)) {
  G <- retorno_descontado(traj$recompensas, g)
  cat(sprintf("  gamma = %.2f  ->  G_0 = %8.4f\n", g, G))
}

cat("\nObsérvese que con gamma=1 recuperamos el retorno no descontado (",
    traj$retorno_no_descontado,
    "),\nmientras que con gamma=0 sólo cuenta la primera recompensa (",
    traj$recompensas[1], ").\n", sep = "")

## 4. Análisis estadístico de retornos bajo política aleatoria

Una sola trayectoria nos dice poco. La cantidad de interés teórica es el **valor de la política**:
$$
J(\pi) \;=\; \mathbb{E}_{\tau \sim \pi}[\, G_0 \,].
$$

No disponiendo de la dinámica $P$ en forma cerrada, **estimamos** $J(\pi)$ por el método de Monte Carlo: simulamos $N$ episodios independientes bajo la misma política y promediamos los retornos. Este es, en esencia, el primer algoritmo de RL que veremos —de manera más sistemática— en la Clase 3.

Por la **ley fuerte de los grandes números**, si los retornos $G_0^{(1)}, \ldots, G_0^{(N)}$ son i.i.d. con media finita,
$$
\hat{J}_N(\pi) \;=\; \frac{1}{N} \sum_{i=1}^{N} G_0^{(i)} \;\xrightarrow{c.s.}\; J(\pi).
$$

El **teorema central del límite** nos permite cuantificar la incertidumbre del estimador mediante un intervalo de confianza asintótico al 95 %.

In [ ]:
N_EPISODIOS <- 500L
retornos    <- numeric(N_EPISODIOS)
longitudes  <- integer(N_EPISODIOS)

set.seed(SEMILLA)
for (i in seq_len(N_EPISODIOS)) {
  semilla_i <- sample.int(.Machine$integer.max, 1L)
  traj_i <- rollout(env, politica_aleatoria, semilla = semilla_i)
  retornos[i]   <- traj_i$retorno_no_descontado
  longitudes[i] <- traj_i$longitud
}

# Estadísticas descriptivas
media   <- mean(retornos)
desv    <- sd(retornos)
ic95    <- 1.96 * desv / sqrt(N_EPISODIOS)
mediana <- median(retornos)

cat(sprintf("=== Estimación de J(pi_rand) sobre %d episodios ===\n\n", N_EPISODIOS))
cat(sprintf("  J_N estimado    = %.3f\n",            media))
cat(sprintf("  IC 95%%         = [%.3f, %.3f]\n",   media - ic95, media + ic95))
cat(sprintf("  Desv. estándar  = %.3f\n",            desv))
cat(sprintf("  Mín / Máx       = %.0f / %.0f\n",     min(retornos), max(retornos)))
cat(sprintf("  Mediana         = %.1f\n",            mediana))

### 4.1 Visualización: distribución empírica y convergencia

In [ ]:
library(patchwork)  # Para combinar dos paneles ggplot

dat_ret <- data.frame(retorno = retornos)

# --- Panel izquierdo: histograma ---
p1 <- ggplot(dat_ret, aes(x = retorno)) +
  geom_histogram(bins = 30, fill = PALETA["navyLight"],
                 color = "white", alpha = 0.85) +
  geom_vline(xintercept = media,   color = PALETA["orange"],
             linewidth = 1, linetype = "solid") +
  geom_vline(xintercept = mediana, color = "#2E8A99",
             linewidth = 1, linetype = "dashed") +
  annotate("label", x = media,   y = Inf, vjust = 1.5,
           label = sprintf("Media = %.1f", media),
           color = PALETA["orange"], fill = "white", size = 3.5) +
  annotate("label", x = mediana, y = Inf, vjust = 3.2,
           label = sprintf("Mediana = %.1f", mediana),
           color = "#2E8A99", fill = "white", size = 3.5) +
  labs(title    = "Distribución empírica de retornos",
       subtitle = "bajo política aleatoria",
       x        = expression("Retorno"~G[0]),
       y        = "Frecuencia") +
  theme_minimal(base_size = 11) +
  theme(plot.title    = element_text(color = PALETA["navy"], face = "bold"),
        plot.subtitle = element_text(color = PALETA["navy"]))

# --- Panel derecho: convergencia del estimador ---
medias_acum <- cumsum(retornos) / seq_len(N_EPISODIOS)
dat_conv <- data.frame(N = seq_len(N_EPISODIOS), media_acum = medias_acum)

p2 <- ggplot(dat_conv, aes(x = N, y = media_acum)) +
  geom_line(color = PALETA["navy"], linewidth = 0.6) +
  geom_hline(yintercept = media, color = PALETA["orange"],
             linewidth = 0.6, linetype = "dashed") +
  annotate("label", x = N_EPISODIOS, y = media, hjust = 1, vjust = -0.5,
           label = sprintf("Media final = %.2f", media),
           color = PALETA["orange"], fill = "white", size = 3.5) +
  labs(title = "Convergencia del estimador Monte Carlo",
       x     = expression("Número de episodios"~N),
       y     = expression(hat(J)[N](pi))) +
  theme_minimal(base_size = 11) +
  theme(plot.title = element_text(color = PALETA["navy"], face = "bold"))

options(repr.plot.width = 11, repr.plot.height = 4.5)
p1 + p2

### 4.2 Lectura estadística

Tres observaciones merecen comentario.

**Primera: asimetría positiva.** El histograma exhibe sesgo a la derecha. La mayoría de los episodios fracasan rápidamente, pero ocasionalmente la dinámica casual del sistema y las acciones aleatorias coinciden en mantener el péndulo en pie por mayor tiempo. La distribución dista de ser gaussiana, hecho relevante al aplicar inferencia clásica: un test que asuma normalidad sin verificación previa puede arrojar conclusiones erróneas.

**Segunda: variabilidad considerable.** La desviación estándar es del orden de la propia media. Esto evidencia que las políticas aleatorias son intrínsecamente ruidosas. Cualquier estimación con $N$ pequeño será imprecisa, y los intervalos de confianza, anchos.

**Tercera: convergencia visible.** La curva acumulada de $\hat{J}_N$ se estabiliza con cientos de episodios. Para alcanzar precisión razonable necesitamos —en este problema concreto— al menos varios cientos de simulaciones, anticipo de un fenómeno general en RL: la **eficiencia de muestra** será una métrica central a lo largo del curso. Algoritmos modernos compiten ferozmente por reducir la cantidad de interacciones agente–entorno necesarias para alcanzar políticas competentes.

## 5. Anticipo: hacia el control óptimo

¿Cuán lejos está $\pi_{\text{rand}}$ del rendimiento óptimo? Con políticas razonables, CartPole-v1 admite retornos de hasta $500$ —el límite por truncamiento—. Nuestra política aleatoria, en cambio, obtiene típicamente entre $20$ y $25$ pasos. La brecha es enorme, y precisamente esa brecha justifica el resto del curso.

Esto motiva las preguntas centrales de la **Clase 2**:

1. ¿Cómo definimos formalmente la **función de valor** $V^\pi(s)$ asociada a una política?
2. ¿Cómo se vincula el valor con la **ecuación de Bellman** —y por qué dicha ecuación admite una solución única en condiciones razonables?
3. Cuando el espacio de estados-acciones es pequeño, ¿podemos enumerar y evaluar políticas de forma sistemática?
4. En problemas más simples (un solo estado, varias acciones), ¿cómo gestionamos el dilema **exploración–explotación**? Aquí emergerá el **bandido multibrazo**.

> **Tarea conceptual sugerida.** Antes del próximo encuentro, reflexione sobre lo siguiente: si dispusiera de la dinámica $P$ exacta del CartPole, ¿podría calcular $V^{\pi_{\text{rand}}}$ analíticamente? ¿Qué dificultad fundamental impide hacerlo en problemas reales?

## Ejercicios propuestos

> Estos ejercicios son optativos en la Clase 1, pero su resolución se valora en la participación. Plantee dudas en el foro de consultas.

### Ejercicio 1 — Estocasticidad en GridWorld

Modifique la clase `GridWorld` para introducir transiciones estocásticas: con probabilidad $p = 0.1$, la acción del agente "resbala" y se ejecuta una acción aleatoria de las cuatro posibles. Verifique que la probabilidad empírica $\hat{P}(s' \mid s, a)$, estimada sobre 10 000 transiciones desde un mismo $(s, a)$, converge a la probabilidad teórica.

### Ejercicio 2 — Retornos descontados y horizonte efectivo

Para CartPole-v1 bajo $\pi_{\text{rand}}$, estime $\mathbb{E}[G_0]$ con $N = 1000$ episodios para $\gamma \in \{0{,}5;\, 0{,}9;\, 0{,}95;\, 0{,}99;\, 1{,}0\}$. Reporte una tabla con media e intervalo de confianza al 95 %. Comente cómo varía la magnitud y la varianza del estimador con $\gamma$.

### Ejercicio 3 — Una política heurística simple

Diseñe una política heurística para CartPole basada únicamente en el ángulo del péndulo: si $\theta > 0$ empuje a la derecha; de lo contrario, a la izquierda. Estime $J(\pi_{\text{heuristica}})$ con $N = 500$ episodios. ¿Mejora respecto de la política aleatoria? Cuantifique la diferencia con un test estadístico apropiado (por ejemplo, t de Welch mediante `t.test()` en R).

### Ejercicio 4 — Análisis de sesgo

¿Por qué fijamos `seed=SEMILLA` solamente en `reset()` y no en cada llamada a `step()`? ¿Qué sucedería si fijásemos la misma semilla en cada episodio? Justifique en términos de la independencia entre realizaciones requerida por el estimador Monte Carlo.

---

### Material complementario

- Sutton & Barto, *Reinforcement Learning: An Introduction* (2.ª ed., 2018), Cap. 1 y 3.1–3.4.
- Gymnasium, [Documentación oficial](https://gymnasium.farama.org/).
- `reticulate`, [Guía de uso](https://rstudio.github.io/reticulate/).
- Repositorio del curso: notebooks Python y R, presentaciones, foro de consultas.

**Próximo encuentro:** martes 12 de mayo de 2026 — *Políticas, funciones de valor y el bandido multibrazo*.

In [ ]:
# Cierre: liberación de recursos
env$close()
cat("Sesión finalizada limpiamente.\n")